# CAJAL Computational Mini-Project 16

## Level 2 — Exploring effect of transcription factor combinations on neuronal cell type composition.

---

You have been given a single-cell RNA-seq dataset from a large-scale combinatorial morphogen perturbation screen. The dataset was generated using Parse single cell technology and there are 2 plates and 192 perturbations in total. You have now analysed and annotated the perturbation screen data with the types of neuronal cells generated in the screen.

**Important:** Do not look up the source paper for this dataset yet.


In [ ]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad

# Required when obs/var contain pandas StringArray columns.
ad.settings.allow_write_nullable_strings = False
pd.options.future.infer_string = False

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(5, 4))
sns.set_style("whitegrid")

%matplotlib inline

In [ ]:
# ---- category display orders -------------------------------------------------
AP_ORDER = ["ctrl",
            "XAV_1", "XAV_2", "XAV_3",
            "CHIR_1", "CHIR_2", "CHIR_3", "CHIR_4",
            "RA_1", "RA_2", "RA_3", "RA_4",
            "RA_1_CHIR", "RA_2_CHIR", "RA_3_CHIR", "RA_4_CHIR",
            "FGF8_1", "FGF8_2", "FGF8_3", "FGF8_4",
            "FGF8_1_CHIR", "FGF8_2_CHIR", "FGF8_3_CHIR", "FGF8_4_CHIR"]
DV_ORDER = ["ctrl", "BMP4_1", "BMP4_2", "BMP4_3",
            "SHH_1", "SHH_2", "SHH_3", "SHH_4"]
REGION_ORDER = ["Forebrain", "Midbrain", "Hindbrain", "Spinal cord",
                "SYM", "ENS", "TG", "DRG"]
NEURON_ORDER = ["GLUT", "CHO", "NOR", "NBL"]

# ---- colours
REGION_COLORS = {"Forebrain": "#fe9b00", "Midbrain": "#f4c40f",
                 "Hindbrain": "#d8443c", "Spinal cord": "#9b3441",
                 "SYM": "#268a8a", "ENS": "#226a99",
                 "TG": "#383b81", "DRG": "#92c051"}
NEURON_COLORS = {"GLUT": "#fe9b00", "CHO": "#33A02C",
                 "NOR": "#383b81", "NBL": "#c9bba2"}

# AP / DV levels graded light -> dark within each morphogen group
AP_COLORS = {
    "ctrl": "#E5E5E5",
    "XAV_1": "#f9b4c9", "XAV_2": "#d8527c", "XAV_3": "#9a133d",
    "CHIR_1": "#dec5da", "CHIR_2": "#b695bc", "CHIR_3": "#90719f", "CHIR_4": "#574571",
    "RA_1": "#aadce0", "RA_2": "#72bcd5", "RA_3": "#528fad", "RA_4": "#376795",
    "RA_1_CHIR": "#c2d6a4", "RA_2_CHIR": "#9cc184", "RA_3_CHIR": "#669d62", "RA_4_CHIR": "#3c7c3d",
    "FGF8_1": "#FFBBFF", "FGF8_2": "#EE7AE9", "FGF8_3": "#B452CD", "FGF8_4": "#8B008B",
    "FGF8_1_CHIR": "#CCCCCC", "FGF8_2_CHIR": "#999999", "FGF8_3_CHIR": "#666666", "FGF8_4_CHIR": "#333333"}
DV_COLORS = {
    "ctrl": "#E5E5E5",
    "BMP4_1": "#ffe6b7", "BMP4_2": "#ffd353", "BMP4_3": "#ffb242",
    "SHH_1": "#C2D9F7", "SHH_2": "#98C1F0", "SHH_3": "#4782DD", "SHH_4": "#1D52A1"}

## 1. Setup and Data Loading

In [ ]:
adata = sc.read_h5ad("data/ngn2_scrna_processed.h5ad", backed="r")
adata.obs_names = adata.obs_names + "_" + adata.obs["plateID"].astype(str)
obs = adata.obs

obs[["parse_id", "AP_axis", "DV_axis", "Region", "Neuron_type"]].head()

## 2. Data exploration

We want to understand the effect of the different transcription factor combination and concentration on the types of neuron generated. One way to approach this is to look at the composition of neuronal cell (e.g. by region, neuron type and/or division) for each morphogen combination. Hint: you may need to use `pd.crosstab` with `normalize="index"` counts the cells of each category in every sample and turns each row into fractions that sum to 1.

### 2.1. Neuronal region composition

We also need to know which treatment each sample got. Since every sample has a
single `AP_axis` and `DV_axis` value, we can grab one row per sample.

To make the plot result easier to interpret, we will use the predefined order for the AP and DV axis above.

Now we can visualise the the region composition per sample. Since we have 192 samples, it will be rather difficult to interpret a bar chart with all 192 samples, especially if there is no clear legend. As such, let's split the plot by DV condition since there is less DV condition compared to AP condition. Start by creating a stack bar chart for one DV condition (e.g. ctrl).

Now generate this for all DV combination, with each plot being a subplot.

While the stacked barplot visualisation may be useful to quickly glance at the regional composition of the cell, it is a bit hard to compare the fraction of cells per region across samples. In this case, something like a boxplot showing the distribution of cell fraction might be helpful.

You should use seaborn `catplot` to generate the boxplots. However, seaborn expect the data to be in a 'tidy' format, with one row per (sample, category), with a `freq` column. `melt` does exactly that. We attach the treatment columns first.

### 2.2. Neuronal type composition

We will focus on the neuronal type and we will generate similar plots as above.

Now comes the biological intepretation. Have a look at the plots you have generated and explore the effect of the morphogens on the regional identity and neuronal type being generated. Here are some questions to get you started:

* Which AP treatments increase **Hindbrain** identity?
* Under which DV treatment do **NOR** neurons appear?
* What dominates the **control** samples?

----

## 3. Replicating main figures from paper

So far, we have analysed the effect of the perturbation at the cell level. While this is useful, this does not necessarily tell us the diversity of cell types generated by the morphogen screen. Similarly, in the paper from which this data came from, the authors focused primarily on the number and types of clusters (as a proxy for cell types) generated by the morphogen screen in the main figures.

Before we reveal the source of the paper and show you the figures the authors have generated, can you come up with ideas of how to show the diversity of cell types (i.e. clusters) for each morphogen combinations?

Now that you have come up and created some figures to show the cell type diversity, let's look at the figures from the [paper](https://www.science.org/doi/10.1126/science.adn6121). In particular, we will focus on figure 3 panel A-C which summarises the result of the morphogen screen.

### 3.1 Panel A

### 3.2 Panel B

### 3.3 Panel C